# DDI Atomic Triplet Extraction — v4

## Root-cause diagnosis of v3 failures

v3 produced only **58 triplets from 50 responses** (26 responses = zero triplets, ~1.2 triplets/response).  
Three failure modes were identified:

| Failure | Example | Root Cause |
|---|---|---|
| **Pronoun subject** | `"it doesn't interact with lamivudine"` | `it` is the grammatical nsubj — no drug token on subj side, verb-centric extraction fires nothing |
| **OOV drugs** | `"Fibrates"`, `"HMG-CoA reductase inhibitors"`, `"ethanol"` | Not in FDA NDC PhraseMatcher → invisible, sentence drops below 2-drug threshold |
| **Self-pairs** | e1 == e2 (degenerate rows) | Nothing to extract, expected |

## v4 fixes: three-tier extraction

Since we are in a **supervised NLI setting** — every holistic response has a known reference pair  
(`e1_text`, `e2_text`) from the DDI-2013 corpus — we exploit this to fix both failure modes:

**Tier 1 — Explicit (both drugs appear as tokens):** Same verb-centric extraction as v3.  
**Tier 2 — OOV fix:** `e1_text` and `e2_text` are injected as per-row matcher patterns,  
guaranteeing recognition even for drug class names and trade names not in the FDA NDC index.  
**Tier 3 — Pronoun coreference:** When a sentence has exactly 1 matched drug token and  
its governing verb's grammatical subject is a pronoun (`it`, `they`, `its`, `their`, `this`),  
the pronoun is resolved to the known partner drug from the reference pair. The verb and  
negation are extracted normally.

The `source_tier` column records which tier produced each triplet for interpretability.

In [1]:
import pandas as pd
import spacy
from spacy.matcher import PhraseMatcher
from collections import defaultdict
from tqdm.notebook import tqdm

In [2]:
df_holistic = pd.read_csv("synthetic_rag_TEST.csv")
df_raw_fda  = pd.read_csv("drug_products.csv", encoding="ISO-8859-1")

nlp = spacy.load("en_core_web_sm")

# ── Build global FDA drug name set ──────────────────────────────────────────
df_prescription = df_raw_fda[df_raw_fda["PRODUCTTYPENAME"] == "HUMAN PRESCRIPTION DRUG"]

fda_drug_names = set()
for col in ["PROPRIETARYNAME", "NONPROPRIETARYNAME", "SUBSTANCENAME"]:
    if col in df_prescription.columns:
        for val in df_prescription[col].dropna().unique():
            val_clean = str(val).lower().strip()
            for part in val_clean.split(";"):
                fda_drug_names.add(part.strip())

global_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
global_patterns = [nlp.make_doc(t) for t in fda_drug_names if len(t.strip()) > 2]
global_matcher.add("FDA_NDC_INDEX", global_patterns)

print(f"Global drug vocabulary: {len(fda_drug_names):,} entries")

Global drug vocabulary: 8,165 entries


In [3]:
# ── Discourse/copular verbs — carry NO DDI information ───────────────────────
DISCOURSE_VERBS = {
    "be", "say", "find", "read", "look", "talk", "go", "come",
    "know", "seem", "use", "make", "get", "have", "think", "check",
    "tell", "show", "mean", "note", "mention", "see", "hear",
    "explain", "describe", "share", "post", "write", "ask",
}

# Pronouns that signal coreference to the contextual drug subject
SUBJECT_PRONOUNS = {"it", "its", "they", "their", "them", "this", "that"}

# dobj nouns with no pharmacological value
NOISE_DOBJS = {
    "stuff", "info", "information", "thing", "something", "study",
    "interaction", "data", "result", "report", "paper", "article",
    "drug", "medication", "med", "it", "they", "this", "that",
}


def build_row_matcher(e1: str, e2: str):
    """
    Build a per-row PhraseMatcher that combines the global FDA NDC index
    with the known reference pair (e1, e2).

    This ensures drug class names ('Fibrates', 'HMG-CoA reductase inhibitors'),
    non-prescription substances ('ethanol'), and trade names ('ZETIA') are
    always recognised even when absent from the FDA NDC index.
    """
    row_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
    # Re-add all global patterns
    row_matcher.add("FDA_NDC_INDEX", global_patterns)

    # Inject the known pair as guaranteed patterns
    extra = set()
    for name in [e1, e2]:
        if name and len(name.strip()) > 2:
            extra.add(name.lower().strip())
    if extra:
        extra_patterns = [nlp.make_doc(n) for n in extra]
        row_matcher.add("KNOWN_PAIR", extra_patterns)

    return row_matcher, {n.lower().strip() for n in [e1, e2] if n and len(n.strip()) > 2}


def governing_verb(token):
    """Walk up the dep tree to the nearest VERB/AUX ancestor."""
    current = token.head
    visited = set()
    while current.i not in visited:
        visited.add(current.i)
        if current.pos_ in ("VERB", "AUX"):
            return current
        if current.head.i == current.i:
            break
        current = current.head
    return None


def expand_conjuncts(token):
    """Return {token} ∪ all conj-siblings sharing the same head."""
    group = {token}
    for child in token.head.children:
        if child.dep_ == "conj" and child != token:
            group.add(child)
    if token.dep_ == "conj":
        group.add(token.head)
        for sibling in token.head.children:
            if sibling.dep_ == "conj":
                group.add(sibling)
    return group


def build_relation(verb_tok, preposition=None):
    """
    Clean, lemmatised relation string: [not] + verb_lemma [+ dobj_noun] [+ prep]
    """
    neg   = any(c.dep_ == "neg" for c in verb_tok.children)
    lemma = verb_tok.lemma_.lower()

    dobj_str = ""
    for child in verb_tok.children:
        if child.dep_ == "dobj" and child.pos_ == "NOUN":
            candidate = child.lemma_.lower()
            if candidate not in NOISE_DOBJS:
                dobj_str = candidate
            break

    parts = []
    if neg:
        parts.append("not")
    parts.append(lemma)
    if dobj_str:
        parts.append(dobj_str)
    if preposition:
        parts.append(preposition.lower())

    return " ".join(parts)


print("Helpers defined.")

Helpers defined.


In [4]:
SUBJ_DEPS = {"nsubj", "nsubjpass", "csubj", "agent"}
OBJ_DEPS  = {"dobj", "pobj", "attr", "oprd"}


def extract_explicit(sent_doc, drug_tokens, drug_idx):
    """
    TIER 1: Both drugs appear as matched tokens in this sentence.
    Verb-centric extraction — same logic as v3 but applied via helper.
    Returns list of (subj, relation, obj, 'tier1') tuples.
    """
    triplets = []
    verb_groups = defaultdict(lambda: {"verb": None, "subj": [], "obj": []})

    for drug_tok in drug_tokens:
        gov = governing_verb(drug_tok)
        if gov is None:
            continue
        verb_groups[gov.i]["verb"] = gov
        dep = drug_tok.dep_

        if dep in SUBJ_DEPS:
            verb_groups[gov.i]["subj"].append(drug_tok)
        elif dep in OBJ_DEPS:
            verb_groups[gov.i]["obj"].append(drug_tok)
        else:
            # positional fallback
            if drug_tok.i > gov.i:
                verb_groups[gov.i]["obj"].append(drug_tok)
            else:
                verb_groups[gov.i]["subj"].append(drug_tok)

    def conj_groups(toks):
        groups, covered = [], set()
        for t in toks:
            if t.i in covered:
                continue
            grp_set = expand_conjuncts(t)
            covered |= {tt.i for tt in grp_set}
            groups.append([tt for tt in grp_set if tt.i in drug_idx])
        return groups

    for verb_i, grp in verb_groups.items():
        verb_tok   = grp["verb"]
        subj_drugs = grp["subj"]
        obj_drugs  = grp["obj"]

        if verb_tok.lemma_.lower() in DISCOURSE_VERBS:
            continue
        if not subj_drugs or not obj_drugs:
            continue

        # Linking preposition
        linking_prep = None
        for child in verb_tok.children:
            if child.dep_ == "prep":
                prep_subtree = {t.i for t in child.subtree}
                if any(od.i in prep_subtree for od in obj_drugs):
                    linking_prep = child.text.lower()
                    break

        relation = build_relation(verb_tok, linking_prep)

        for sg in conj_groups(subj_drugs):
            for og in conj_groups(obj_drugs):
                s_names = {t.text.lower() for t in sg if t.i in drug_idx}
                o_names = {t.text.lower() for t in og if t.i in drug_idx}
                if not s_names or not o_names:
                    continue
                if s_names & o_names:  # intra-conjunct guard
                    continue
                s_name = sorted(s_names)[0]
                for o_name in sorted(o_names):
                    if s_name != o_name:
                        triplets.append((s_name, relation, o_name, "tier1"))

    # Fallback within tier 1: root verb, all drug pairs
    if not triplets:
        root = [t for t in sent_doc if t.dep_ == "ROOT"]
        if root and root[0].pos_ in ("VERB", "AUX") and root[0].lemma_.lower() not in DISCOURSE_VERBS:
            verb_tok = root[0]
            relation = build_relation(verb_tok, None)
            sorted_drugs = sorted(drug_tokens, key=lambda t: t.i)
            for i in range(len(sorted_drugs)):
                for j in range(i + 1, len(sorted_drugs)):
                    d1 = sorted_drugs[i].text.lower()
                    d2 = sorted_drugs[j].text.lower()
                    if d1 != d2:
                        triplets.append((d1, relation, d2, "tier1_fallback"))

    return triplets


def extract_pronoun_coreference(sent_doc, drug_tokens, known_pair):
    """
    TIER 3: Exactly 1 drug token found in this sentence AND the governing
    verb's grammatical subject is a pronoun (it/they/this/their/that).

    The pronoun is resolved to the partner drug in `known_pair`.
    e.g. 'it doesn't interact with lamivudine' + known_pair={abacavir, lamivudine}
         → (abacavir, not interact with, lamivudine)

    This handles the dominant failure mode in casual AI-generated DDI text,
    which frequently uses 'it' to refer back to the primary drug entity.
    """
    triplets = []
    if len(drug_tokens) != 1:
        return triplets

    known_pair_list = sorted(known_pair)  # stable order
    found_drug = drug_tokens[0].text.lower()
    partner_drugs = [d for d in known_pair_list if d != found_drug]
    if not partner_drugs:
        return triplets  # self-pair or no partner
    partner = partner_drugs[0]

    drug_tok = drug_tokens[0]
    gov      = governing_verb(drug_tok)
    if gov is None or gov.lemma_.lower() in DISCOURSE_VERBS:
        return triplets

    # Check if the verb has a pronominal subject
    pronoun_subj = None
    for child in gov.children:
        if child.dep_ in ("nsubj", "nsubjpass") and child.text.lower() in SUBJECT_PRONOUNS:
            pronoun_subj = child
            break

    if pronoun_subj is None:
        # Also check: drug is the object-side and subject might be elided
        # (passive constructions: 'abacavir is not affected by [it=ethanol]')
        # In this case drug is nsubjpass, check if any prep child has a pronoun pobj
        if drug_tok.dep_ in ("nsubjpass",):
            for child in gov.children:
                if child.dep_ == "agent":
                    for grandchild in child.children:
                        if grandchild.dep_ == "pobj" and grandchild.text.lower() in SUBJECT_PRONOUNS:
                            pronoun_subj = grandchild
                            break

    if pronoun_subj is None:
        return triplets

    # Determine which side the found drug is on
    dep = drug_tok.dep_
    linking_prep = None
    for child in gov.children:
        if child.dep_ == "prep":
            if drug_tok.i in {t.i for t in child.subtree}:
                linking_prep = child.text.lower()
                break

    relation = build_relation(gov, linking_prep)

    if dep in SUBJ_DEPS:
        # Drug is on subj side — partner (pronoun referent) is the object
        triplets.append((found_drug, relation, partner, "tier3_pronoun"))
    else:
        # Drug is on obj side — partner (pronoun referent) is the subject
        triplets.append((partner, relation, found_drug, "tier3_pronoun"))

    return triplets


print("Extraction tiers defined.")

Extraction tiers defined.


In [6]:
# ── Main loop ────────────────────────────────────────────────────────────────
atomic_rows = []

# Added tqdm so you can see progress
for _, row in tqdm(df_holistic.iterrows(), total=len(df_holistic)):
    
    # 🔴 CHANGED: Mapping to the new synthetic output column
    text    = row["synthetic_rag_output"]
    premise = row["premise"]
    e1      = str(row["e1_text"]).strip() if pd.notna(row["e1_text"]) else ""
    e2      = str(row["e2_text"]).strip() if pd.notna(row["e2_text"]) else ""

    if pd.isna(text) or not text.strip():
        continue

    # Skip degenerate self-pairs
    if e1.lower() == e2.lower():
        continue

    # TIER 2 FIX: build a per-row matcher that guarantees e1 and e2 are recognised
    row_matcher, known_pair = build_row_matcher(e1, e2)

    doc = nlp(text)
    
    # Track if ANY triplets were found for this generation
    found_triplets = False 

    for sentence in doc.sents:
        sent_doc = nlp(sentence.text)

        # Match drugs in this sentence (using the row-level matcher)
        matches = row_matcher(sent_doc)
        seen_starts = {}
        for match_id, start, end in matches:
            if start not in seen_starts:
                seen_starts[start] = sent_doc[start]

        # Deduplicate by text
        seen_names = {}
        for tok in seen_starts.values():
            name = tok.text.lower()
            if name not in seen_names:
                seen_names[name] = tok
        drug_tokens = list(seen_names.values())
        drug_idx    = {t.i for t in drug_tokens}

        triplets = []

        if len(drug_tokens) >= 2:
            # TIER 1: explicit — both drugs present
            triplets = extract_explicit(sent_doc, drug_tokens, drug_idx)

        elif len(drug_tokens) == 1:
            # TIER 3: pronoun coreference — one drug found, look for pronoun subject
            triplets = extract_pronoun_coreference(sent_doc, drug_tokens, known_pair)

        for subj, rel, obj, tier in triplets:
            if not rel.strip():
                continue
            found_triplets = True
            atomic_rows.append({
                "original_id":  row["original_id"], # 🔴 ADDED
                "Entity1":      e1,
                "Entity2":      e2,
                "scenario":     row["scenario"],    # 🔴 ADDED,
                "premise":      premise,
                "holistic_generation": text,
                "sub_extract":  subj,
                "obj_extract":  obj,
                "rel_extract":  rel,
                "source_tier":         tier,
            })
            
    # 🔴 ADDED: If the fake_drug scenario caused the parser to find NOTHING (Dictionary Failsafe)
    if not found_triplets:
        atomic_rows.append({
            "original_id":  row["original_id"],
            "Entity1":      e1,
            "Entity2":      e2,
            "scenario":     row["scenario"],
            "premise":      premise,
            "holistic_generation": text,
            "sub_extract":  None,
            "obj_extract":  None,
            "rel_extract":  None,
            "source_tier":  "filtered_out", # Flags that scispaCy rejected it
        })

df_atomic = pd.DataFrame(atomic_rows)
print(f"Total rows logged (including filtered) : {len(df_atomic):,}")
print(f"Holistic responses       : {len(df_holistic)}")
print(f"Avg triplets per response: {len(df_atomic.dropna(subset=['rel_extract']))/len(df_holistic):.1f}")
print()
print("Tier breakdown:")
print(df_atomic["source_tier"].value_counts().to_string())

# Verification print to see if the Fake Drug scenario was successfully filtered
print("\nSuccess Check: Did the failsafe catch the fake drug scenarios?")
print(df_atomic.groupby("scenario")["rel_extract"].count().to_string())

  0%|          | 0/80 [00:00<?, ?it/s]

Total rows logged (including filtered) : 277
Holistic responses       : 80
Avg triplets per response: 3.1

Tier breakdown:
source_tier
tier1_fallback    207
tier1              39
filtered_out       28
tier3_pronoun       3

Success Check: Did the failsafe catch the fake drug scenarios?
scenario
contradiction     20
entailment        90
fake_drug        122
neutral           17


In [7]:
print("Relation distribution:")
print(df_atomic["rel_extract"].value_counts().head(30).to_string())
print()
print("Coverage — responses with ≥1 triplet:")
covered = df_atomic["holistic_generation"].nunique()
print(f"  {covered} / {len(df_holistic)} responses ({100*covered/len(df_holistic):.0f}%)")
print()
print("Sample triplets:")
df_atomic[["Entity1","Entity2","sub_extract","rel_extract","obj_extract","source_tier"]].head(20)

Relation distribution:
rel_extract
interact                    65
study                       45
conduct                     36
increase level              11
enhance effect              10
include antibiotic          10
lead                         8
belong                       6
alter parameter              5
inhibit metabolism           4
result                       3
result in                    3
decrease bioavailability     3
work                         3
decrease                     2
require monitoring           2
play role                    2
interact with                2
impair absorption            2
not alter parameter          2
not recommend                1
not administer               1
increase risk                1
not increase risk            1
target clot                  1
exhibit                      1
decrease effectiveness       1
enhance absorption           1
involve                      1
occur                        1

Coverage — responses with ≥1 tripl

,Entity1,Entity2,sub_extract,rel_extract,obj_extract,source_tier
0,thioxanthines,Permax,metoclopramide,not recommend,permax,tier1_fallback
1,thioxanthines,Permax,NaN,NaN,NaN,filtered_out
2,thioxanthines,Permax,NaN,NaN,NaN,filtered_out
3,thioxanthines,Permax,thioxanthines,not administer,metoclopramide,tier1_fallback
4,Streptase,antiplatelet agents,antiplatelet,increase risk,streptase,tier1
5,Streptase,antiplatelet agents,antiplatelet,not increase risk,streptase,tier1
6,Streptase,antiplatelet agents,streptase,target clot,antiplatelet,tier1_fallback
7,Streptase,antiplatelet agents,NaN,NaN,NaN,filtered_out
8,Vitamin B1,Loop Diuretics,vitamin,decrease,loop,tier1
9,Vitamin B1,Loop Diuretics,thiamine,decrease,loop,tier1


In [8]:
df_atomic.to_csv("extracted_atomic_triplets_v4.csv", index=False)
print("Saved to extracted_atomic_triplets_v4.csv")

Saved to extracted_atomic_triplets_v4.csv
